In [1]:
import os
import pickle
from pathlib import Path
import numpy as np
import soundfile as sf
from tqdm import tqdm

In [ ]:
DATA_DIR = Path("C:\Users\Сичкаренко\autodition\data")
RAW_DIR = DATA_DIR / "raw" / "source_sep_formatted"
PREPROCESSED_DIR = DATA_DIR / "preprocessed" / "source_sep_formatted"

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_audio(path):
    audio, sr = sf.read(path)
    return audio, sr

# --- Cell 4: Feature Extraction ---
def compute_features(audio, sr):
    # SIMPLE baseline: use raw waveform
    # You can replace with spectrogram later
    return audio.astype(np.float32)

# --- Cell 5: Target Processing ---
def load_sources(sources_dir):
    sources = []
    for file in sorted(os.listdir(sources_dir)):
        if file.endswith(".wav"):
            audio, _ = load_audio(sources_dir / file)
            sources.append(audio)
    
    # Stack shape: (num_sources, T)
    return np.stack(sources, axis=0)

In [ ]:
features = {}
targets = {}
keys = []

for mix_dir in tqdm(sorted(RAW_DIR.iterdir())):
    if not mix_dir.is_dir():
        continue

    mix_id = mix_dir.name
    mixture_path = mix_dir / "mixture.wav"
    sources_dir = mix_dir / "sources"

    if not mixture_path.exists() or not sources_dir.exists():
        continue

    mixture, sr = load_audio(mixture_path)
    sources = load_sources(sources_dir)

    features[mix_id] = compute_features(mixture, sr)
    targets[mix_id] = sources.astype(np.float32)

    keys.append(mix_id)

print(f"Processed {len(keys)} samples")

In [ ]:
np.random.seed(42)
keys = np.array(keys)
np.random.shuffle(keys)

n = len(keys)
train_split = int(0.8 * n)
val_split = int(0.9 * n)

train_keys = keys[:train_split].tolist()
val_keys = keys[train_split:val_split].tolist()
test_keys = keys[val_split:].tolist()

print(len(train_keys), len(val_keys), len(test_keys))


In [ ]:
with open(PREPROCESSED_DIR / "features.pkl", "wb") as f:
    pickle.dump(features, f)

with open(PREPROCESSED_DIR / "targets.pkl", "wb") as f:
    pickle.dump(targets, f)

with open(PREPROCESSED_DIR / "train_keys.pkl", "wb") as f:
    pickle.dump(train_keys, f)

with open(PREPROCESSED_DIR / "val_keys.pkl", "wb") as f:
    pickle.dump(val_keys, f)

with open(PREPROCESSED_DIR / "test_keys.pkl", "wb") as f:
    pickle.dump(test_keys, f)

print("Saved preprocessed dataset!")



In [ ]:

sample_key = train_keys[0]
print("Feature shape:", features[sample_key].shape)
print("Target shape:", targets[sample_key].shape)
